In [3]:
import pandas as pd
from sqlalchemy import create_engine

In [4]:
df = pd.read_csv("../data/telco.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


A useful way to think about the sequence is:
1. Overall KPI
   - What is the churn rate?
2. Segment the problem
   - Which groups churn more?
3. Find patterns
   - Contract type
   - Payment method
   - Tenure
   - Monthly charges
   - Internet service
   - Senior citizen status
4. Quantify impact
   - How much revenue is at risk?
   - How many high-risk customers are there?
5. Build prediction
   - Which individual customers are likely to churn?

In [5]:
df.shape

(7043, 21)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [7]:
df["TotalCharges"].head(10)

0      29.85
1     1889.5
2     108.15
3    1840.75
4     151.65
5      820.5
6     1949.4
7      301.9
8    3046.05
9    3487.95
Name: TotalCharges, dtype: object

In [8]:
(df["TotalCharges"].str.strip() == "").sum()

11

In [9]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [10]:
df[df["TotalCharges"].isna()][
    ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]
]

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,NaN,No
753,3115-CZMZD,0,20.25,NaN,No
936,5709-LVOEQ,0,80.85,NaN,No
1082,4367-NUYAO,0,25.75,NaN,No
1340,1371-DWPAZ,0,56.05,NaN,No
3331,7644-OMVMY,0,19.85,NaN,No
3826,3213-VVOLG,0,25.35,NaN,No
4380,2520-SGTTA,0,20.00,NaN,No
5218,2923-ARZLG,0,19.70,NaN,No
6670,4075-WKNIU,0,73.35,NaN,No


In [11]:
df["TotalCharges"].isna().sum()

11

In [12]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [13]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692,2279.734304
std,0.368612,24.559481,30.090047,2266.794470
min,0.000000,0.000000,18.250000,0.000000
25%,0.000000,9.000000,35.500000,398.550000
50%,0.000000,29.000000,70.350000,1394.550000
75%,0.000000,55.000000,89.850000,3786.600000
max,1.000000,72.000000,118.750000,8684.800000


“I used descriptive statistics to understand the customer population before deeper analysis. The average customer tenure was about 32 months, average monthly charges were around $65, and approximately 16% of customers were senior citizens. These findings helped me identify tenure, pricing, and customer demographics as key areas to investigate for churn.”

In [14]:
df.describe(include="object")

,customerID,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Churn
count,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043
unique,7043,2,2,2,2,3,3,3,3,3,3,3,3,3,2,4,2
top,7590-VHVEG,Male,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,No
freq,1,3555,3641,4933,6361,3390,3096,3498,3088,3095,3473,2810,2785,3875,4171,2365,5174


In [15]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [16]:
df.duplicated().sum()

0

In [17]:
from sqlalchemy import create_engine

engine = create_engine(
    "postgresql+psycopg2://harikasunkara@localhost:5432/telco"
)

In [18]:
df.to_sql(
    "customers",
    engine,
    if_exists="replace",
    index=False
)

print("Data loaded successfully")

Data loaded successfully


In [19]:
pd.read_sql(
    "SELECT COUNT(*) AS total_customers FROM customers;",
    engine
)

,total_customers
0,7043


In [20]:
pd.read_sql(
    "SELECT * FROM customers LIMIT 5;",
    engine
)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [21]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.columns

Index(['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents',
       'tenure', 'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod', 'monthlycharges', 'totalcharges', 'churn'],
      dtype='object')

In [22]:
df.to_sql(
    "customers",
    engine,
    if_exists="replace",
    index=False
)

print("Reloaded customers table with lowercase columns")

Reloaded customers table with lowercase columns


In [23]:
pd.read_sql(
    """
    SELECT
        churn,
        COUNT(*) AS customer_count
    FROM customers
    GROUP BY churn;
    """,
    engine
)

,churn,customer_count
0,No,5174
1,Yes,1869


## “How many customers left, and what percentage of all customers does that represent?”

In [24]:
query = """
SELECT
    COUNT(*) AS total_customers,
    SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate_percentage
FROM customers;
"""

pd.read_sql(query, engine)

,total_customers,churned_customers,churn_rate_percentage
0,7043,1869,26.54


## Which contract type has the highest churn rate?

In [25]:
df['contract'].unique()

array(['Month-to-month', 'One year', 'Two year'], dtype=object)

In [26]:
df['contract'].value_counts()

contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

In [27]:
df['contract'].value_counts(normalize=True) * 100

contract
Month-to-month    55.019168
Two year          24.066449
One year          20.914383
Name: proportion, dtype: float64

In [28]:
pd.read_sql(
    """
        select contract,
        count(*) as total_customers,
        SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate_percentage
    from customers
    group by contract
    ORDER BY churn_rate_percentage DESC;
    """, engine
)

,contract,total_customers,churned_customers,churn_rate_percentage
0,Month-to-month,3875,1655,42.71
1,One year,1473,166,11.27
2,Two year,1695,48,2.83


“Month-to-month customers have the highest churn rate, which suggests contract commitment is strongly associated with retention.”

## “Which payment method has the highest churn rate?”

In [29]:
df['paymentmethod'].unique()

array(['Electronic check', 'Mailed check', 'Bank transfer (automatic)',
       'Credit card (automatic)'], dtype=object)

In [30]:
df['paymentmethod'].value_counts()

paymentmethod
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1544
Credit card (automatic)      1522
Name: count, dtype: int64

In [31]:
df['paymentmethod'].value_counts(normalize=True)*100

paymentmethod
Electronic check             33.579441
Mailed check                 22.887974
Bank transfer (automatic)    21.922476
Credit card (automatic)      21.610109
Name: proportion, dtype: float64

In [32]:
query = """
SELECT
    paymentmethod,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate_percentage
FROM customers
GROUP BY paymentmethod
ORDER BY churn_rate_percentage DESC;
"""

pd.read_sql(query, engine)

,paymentmethod,total_customers,churned_customers,churn_rate_percentage
0,Electronic check,2365,1071,45.29
1,Mailed check,1612,308,19.11
2,Bank transfer (automatic),1544,258,16.71
3,Credit card (automatic),1522,232,15.24


“Electronic check is strongly associated with higher churn and should be investigated further.”

### Are electronic-check customers also mostly month-to-month customers?

In [33]:
query = """
SELECT
    paymentmethod,
    contract,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate_percentage
FROM customers
GROUP BY paymentmethod, contract
ORDER BY paymentmethod, churn_rate_percentage DESC;
"""

pd.read_sql(query, engine)

,paymentmethod,contract,total_customers,churned_customers,churn_rate_percentage
0,Bank transfer (automatic),Month-to-month,589,201,34.13
1,Bank transfer (automatic),One year,391,38,9.72
2,Bank transfer (automatic),Two year,564,19,3.37
3,Credit card (automatic),Month-to-month,543,178,32.78
4,Credit card (automatic),One year,398,41,10.30
5,Credit card (automatic),Two year,581,13,2.24
6,Electronic check,Month-to-month,1850,994,53.73
7,Electronic check,One year,347,64,18.44
8,Electronic check,Two year,168,13,7.74
9,Mailed check,Month-to-month,893,282,31.58


## “Are newer customers more likely to leave than long-term customers? i.e (tenure) for how many months customers stay and do not churn(leave)

In [34]:
query = """
SELECT
    CASE
        WHEN tenure <= 12 THEN '0-12 months'
        WHEN tenure <= 24 THEN '13-24 months'
        WHEN tenure <= 48 THEN '25-48 months'
        ELSE '49-72 months'
    END AS tenure_group,

    COUNT(*) AS total_customers,

    SUM(
        CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END
    ) AS churned_customers,

    ROUND(
        100.0 * SUM(
            CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END
        ) / COUNT(*),
        2
    ) AS churn_rate_percentage

FROM customers

GROUP BY tenure_group

ORDER BY churn_rate_percentage DESC;
"""

pd.read_sql(query, engine)

,tenure_group,total_customers,churned_customers,churn_rate_percentage
0,0-12 months,2186,1037,47.44
1,13-24 months,1024,294,28.71
2,25-48 months,1594,325,20.39
3,49-72 months,2239,213,9.51


## Do churned customers tend to pay higher monthly bills?

In [35]:
#monthly charge vs churn

In [43]:
pd.read_sql_query(
    """
    SELECT
        churn,
        COUNT(*) AS total_customers,
        ROUND(AVG(monthlycharges)::numeric, 2) AS avg_monthly_charge,
        ROUND(MIN(monthlycharges)::numeric, 2) AS lowest_bill,
        ROUND(MAX(monthlycharges)::numeric, 2) AS highest_bill
    FROM customers
    GROUP BY churn
    """, engine
)

,churn,total_customers,avg_monthly_charge,lowest_bill,highest_bill
0,No,5174,61.27,18.25,118.75
1,Yes,1869,74.44,18.85,118.35


In [44]:
query = """
SELECT
    internetservice,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate
FROM customers
GROUP BY internetservice
ORDER BY churn_rate DESC;
"""

pd.read_sql(query, engine)

,internetservice,total_customers,churned_customers,churn_rate
0,Fiber optic,3096,1297,41.89
1,DSL,2421,459,18.96
2,No,1526,113,7.40
